# Trust Chat - Android APK Builder

## 使用步骤

依次点击每个单元格左侧的播放按钮运行即可

In [ ]:
# 第 1 步：安装系统依赖（约 2 分钟）
!sudo apt-get update -qq
!sudo apt-get install -y -qq openjdk-17-jdk git zip unzip autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo5 cmake libffi-dev libssl-dev python3-pip > /dev/null 2>&1
!java -version 2>&1
print("OK - System dependencies installed")

In [ ]:
# 第 2 步：安装 Buildozer（约 1 分钟）
!pip install buildozer -q
!buildozer version 2>&1
print("OK - Buildozer installed")

In [ ]:
# 第 3 步：上传项目文件
# 运行后点击选择文件，上传 project.zip

import os
os.chdir("/content")
%cd /content

# 清理旧文件
!rm -f main.py network.py buildozer.spec project.zip
!rm -rf bin

from google.colab import files
import zipfile

print("Please upload project.zip...")
uploaded = files.upload()

if "project.zip" in uploaded:
    with zipfile.ZipFile("project.zip", "r") as zf:
        zf.extractall("/content")
    print("Extracted OK")
else:
    print("ERROR: Please upload a file named project.zip")

# 验证文件
!echo === Files in /content ===
!ls -la /content/main.py /content/network.py /content/buildozer.spec 2>&1
print("\nIf you see 3 files above, go to Step 4.")

In [ ]:
# 第 4 步：修复 p4a recipe URL 并构建 APK（15-25 分钟）
# 如果构建失败，修完问题后可以重新运行这个 cell

import os, re, json, urllib.request, urllib.error

os.chdir("/content")
os.environ["BUILDOZER_WARN_ON_ROOT"] = "1"

# ============================================================
#  阶段 A：修复 python-for-android 的 recipe 下载 URL
# ============================================================

# A1. 读取 spec 中的 Python minor 版本
with open("/content/buildozer.spec") as f:
    spec = f.read()
ver_match = re.search(r'hostpython3==(\d+\.\d+)', spec)
py_ver = ver_match.group(1) if ver_match else "3.10"
print(f"Target Python version: {py_ver}")

# A2. 确定完整的补丁版本号 (e.g. 3.10 -> 3.10.16)
FALLBACK_VERSIONS = {
    "3.10": "3.10.16",
    "3.11": "3.11.15",
    "3.12": "3.12.11",
    "3.13": "3.13.8",
}
full_version = None

print(f"Probing GitHub for latest {py_ver}.x tag...")
for patch in range(25, 0, -1):
    test_ver = f"{py_ver}.{patch}"
    test_url = f"https://github.com/python/cpython/archive/refs/tags/v{test_ver}.tar.gz"
    try:
        req = urllib.request.Request(test_url, method="HEAD")
        req.add_header("User-Agent", "colab-buildozer")
        with urllib.request.urlopen(req, timeout=10) as resp:
            full_version = test_ver
            print(f"  Found: v{full_version} (confirmed via HEAD)")
            break
    except urllib.error.HTTPError as e:
        if e.code == 404:
            continue
    except Exception as e:
        print(f"  Network error at v{test_ver}: {e}")
        break

if not full_version:
    full_version = FALLBACK_VERSIONS.get(py_ver, f"{py_ver}.11")
    print(f"Using fallback version: {full_version}")
else:
    print(f"Using confirmed version: {full_version}")

# A3. 准备 python-for-android
p4a_dir = "/content/.buildozer/android/platform/python-for-android"
init_py = f"{p4a_dir}/pythonforandroid/recipes/hostpython3/__init__.py"

if os.path.exists(p4a_dir) and not os.path.exists(init_py):
    print(f"Removing incomplete p4a clone...")
    import shutil
    shutil.rmtree(p4a_dir, ignore_errors=True)

if os.path.exists(init_py):
    print(f"p4a already exists, patching in place...")
else:
    print(f"Cloning p4a develop branch...")
    !mkdir -p /content/.buildozer/android/platform
    !git clone --depth 1 -b develop https://github.com/kivy/python-for-android.git {p4a_dir} 2>&1 | tail -3

# A4. 修复 hostpython3 和 python3 recipe 版本号
for recipe_name in ["hostpython3", "python3"]:
    recipe_path = f"{p4a_dir}/pythonforandroid/recipes/{recipe_name}/__init__.py"
    if not os.path.exists(recipe_path):
        print(f"  SKIP {recipe_name}: file not found")
        continue
    with open(recipe_path, 'r') as f:
        original = f.read()
    v_match = re.search(r"version\s*=\s*['\"]([^'\"]+)", original)
    current_ver = v_match.group(1) if v_match else "???"
    if current_ver == full_version:
        print(f"  OK {recipe_name}: already correct")
        continue
    new_content = re.sub(
        r"(version\s*=\s*['\"])[^'\"]*(['\"])",
        rf"\g<1>{full_version}\g<2>", original
    )
    with open(recipe_path, 'w') as f:
        f.write(new_content)
    print(f"  PATCHED {recipe_name}: '{current_ver}' -> '{full_version}'")

# A5. 修复 libthorvg recipe 的 libomp.so glob bug
thorvg_path = f"{p4a_dir}/pythonforandroid/recipes/libthorvg/__init__.py"
if os.path.exists(thorvg_path):
    with open(thorvg_path, 'r') as f:
        thorvg_content = f.read()
    old_pattern = (
        r'pattern = join\(self\.ctx\.ndk\.llvm_prebuilt_dir, '
        r'"lib/clang/\*/lib/linux", lib_arch\)\s*\n'
        r'\s*clang_lib_dir = glob\(pattern\)\[0\]\s*\n'
        r'\s*libomp = join\(clang_lib_dir, "libomp\.so"\)\s*\n'
        r'\s*shprint\(sh\.cp, libomp, join\("install", "lib"\)\)'
    )
    replacement = (
        '# Try multiple patterns to find libomp.so\\n'
        '            patterns = [\\n'
        '                join(self.ctx.ndk.llvm_prebuilt_dir, "lib/clang/*/lib/linux", lib_arch, "libomp.so"),\\n'
        '                join(self.ctx.ndk.llvm_prebuilt_dir, "lib64/clang/*/lib/linux", lib_arch, "libomp.so"),\\n'
        '                join(self.ctx.ndk.llvm_prebuilt_dir, "lib", lib_arch + "-linux-android", "libomp.so"),\\n'
        '            ]\\n'
        '            libomp = None\\n'
        '            for pat in patterns:\\n'
        '                matches = glob(pat)\\n'
        '                if matches:\\n'
        '                    libomp = matches[0]\\n'
        '                    print(f"  Found libomp.so: {libomp}")\\n'
        '                    break\\n'
        '            if libomp:\\n'
        '                shprint(sh.cp, libomp, join("install", "lib"))\\n'
        '            else:\\n'
        '                print("  WARNING: libomp.so not found, skipping OpenMP copy")'
    )
    if re.search(old_pattern, thorvg_content):
        thorvg_content = re.sub(old_pattern, replacement, thorvg_content)
        with open(thorvg_path, 'w') as f:
            f.write(thorvg_content)
        print("  PATCHED libthorvg: libomp.so glob now resilient")
    else:
        print("  WARNING: libthorvg pattern not matched")

# A6. 修复 p4a build.py 的 pip 升级 bug
# pip install -U pip 损坏 vendored resolvelib; 用 get-pip.py 全新安装
build_py_path = f"{p4a_dir}/pythonforandroid/build.py"
if os.path.exists(build_py_path):
    with open(build_py_path, 'r') as f:
        build_py = f.read()
    old_upgrade = '"source venv/bin/activate && pip install -U pip"'
    new_upgrade = (
        '"source venv/bin/activate && '
        'python -c \\"from urllib.request import urlopen;'
        'exec(urlopen(\\'https://bootstrap.pypa.io/get-pip.py\\').read())\\"'
        ' pip==24.0"'
    )
    if old_upgrade in build_py:
        build_py = build_py.replace(old_upgrade, new_upgrade)
        with open(build_py_path, 'w') as f:
            f.write(build_py)
        print("  PATCHED p4a build.py: pip upgrade now uses get-pip.py")
    else:
        print("  WARNING: pip upgrade pattern not matched")
else:
    print("  SKIP build.py: not found")

# ============================================================
#  阶段 B：构建 APK
# ============================================================

print("\n" + "=" * 50)
!echo "=== buildozer.spec settings ==="
!grep -E "^(requirements|p4a\.branch|android\.archs|android\.ndk)" /content/buildozer.spec
print("=" * 50)

print("\nBuilding APK... (15-25 min)\n")
!buildozer android debug --verbose 2>&1

# ============================================================
#  阶段 C：检查结果
# ============================================================

print()
print("=" * 50)
if os.path.exists("bin"):
    apks = sorted([f for f in os.listdir("bin") if f.endswith(".apk")])
    if apks:
        print(f"BUILD SUCCESS! {len(apks)} APK(s):")
        for apk in apks:
            size_mb = os.path.getsize(f"bin/{apk}") / (1024 * 1024)
            print(f"   {apk} ({size_mb:.1f} MB)")
    else:
        print("WARNING: No APK found in bin/")
else:
    print("BUILD FAILED! Check the error above.")

In [ ]:
# 第 5 步：下载 APK

from google.colab import files
import glob, os

os.chdir("/content")

apks = sorted(glob.glob("bin/*.apk"))
if apks:
    apk_path = apks[0]
    size_mb = os.path.getsize(apk_path) / (1024*1024)
    print(f"Downloading: {os.path.basename(apk_path)} ({size_mb:.1f} MB)")
    files.download(apk_path)
    print("\nDONE! Transfer the APK to your phone and install it.")
else:
    print("ERROR: No APK found. The build may have failed in Step 4.")